# Sample 07: モダン宣言的機能 (v0.3.0+)

最新の宣言的機能である、半精度浮動小数点数 (`Float16`)、値域バリデーション (`Range`)、固定総フレームサイズ (`total_size`)、長さ・要素数の自動計算・連動デシリアライズ (`LengthOf`, `CountOf`)、およびインタラクティブ HTML 仕様書生成を学びます。

### 学べる内容
- `Float16` (IEEE 754 16ビット半精度浮動小数点数)
- `Range[Type, min, max]` による宣言的値域バリデーション
- `total_size=16` による固定長フレーム強制と自動パディング
- `LengthOf` / `CountOf` による可変長フィールドの長さ自動計算・連動デシリアライズ
- `BinaryWriter.pad_to()` による特定オフセットへのパディング
- `to_html()` / `write_html()` による双方向 Hex Inspector 付きスタンドアロン HTML 仕様書の生成

In [1]:
from pathlib import Path

from binary_master import (
    Array,
    BinaryWriter,
    Bytes,
    CountOf,
    Float16,
    LengthOf,
    Range,
    RangeValidationError,
    UInt8,
    UInt16,
    binary_struct,
    hexdump,
)

## 1. `Float16`, `Range` バリデーション, および `total_size` パディング

センサーデータブロックなど、サイズが厳格に固定されたフレームを定義します。

In [2]:
@binary_struct(total_size=16, pad_byte=b"\x00")
class EnvironmentalSensorBlock:
    """16バイト固定の環境センサーデータブロック."""
    sensor_id: UInt16
    temperature_c: Range[Float16, -40.0, 85.0]  # [-40, 85] ℃
    humidity_pct: Range[UInt8, 0, 100]          # [0, 100] %
    battery_v: Float16                          # 半精度浮動小数点数

sensor = EnvironmentalSensorBlock(
    sensor_id=0x101,
    temperature_c=25.5,
    humidity_pct=60,
    battery_v=3.3,
)
data = sensor.to_bytes()
print("シリアライズバイナリ (厳格に16バイト):")
print(hexdump(data, annotate=True))
assert len(data) == 16

# デシリアライズ検証
recovered = EnvironmentalSensorBlock.from_bytes(data)
print(f"Recovered Temp:    {float(recovered.temperature_c):.1f} °C")
print(f"Recovered Battery: {float(recovered.battery_v):.2f} V")

# Range 範囲外エラーの検証
try:
    bad_sensor = EnvironmentalSensorBlock(
        sensor_id=0x102,
        temperature_c=120.0,  # 上限 85.0 を超過！
        humidity_pct=50,
        battery_v=3.3,
    )
    bad_sensor.to_bytes()
except RangeValidationError as e:
    print(f"期待通りの RangeValidationError を検知: {e}")

シリアライズバイナリ (厳格に16バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  01 01 60 4e 3c 9a 42 00  00 00 00 00 00 00 00 00  |..`N<.B.........|
  [Total: 16 bytes (`0x0010`)]
Recovered Temp:    25.5 °C
Recovered Battery: 3.30 V
期待通りの RangeValidationError を検知: Field 'temperature_c' value 120.0 is out of valid range [-40.0, 85.0]

## 2. `LengthOf` & `CountOf` (自動長さ計算 & 連動デシリアライズ)

可変長バイト列や可変長配列の手前に配置することで、シリアライズ時に長さを自動計算し、デシリアライズ時にもその長さだけを正確に読み込ませることができます。

In [3]:
@binary_struct
class TelemetryFrame:
    """ペイロード長と要素数が自動計算されるテレメトリフレーム."""
    payload_len: LengthOf[UInt16, "payload"]
    payload: Bytes
    item_count: CountOf[UInt8, "readings"]
    readings: Array[UInt16]
    footer_crc: UInt16

# payload_len と item_count の引数指定は不要！
frame = TelemetryFrame(
    payload=b"GPS_FIX_OK",
    readings=[100, 200, 300, 400],
    footer_crc=0xCAFE,
)

frame_bytes = frame.to_bytes()
print(f"自動計算された payload_len: {frame.payload_len} (期待値: 10 bytes)")
print(f"自動計算された item_count: {frame.item_count} (期待値: 4 items)")
print(f"シリアライズ結果 ({len(frame_bytes)} バイト):")
print(hexdump(frame_bytes, annotate=True))

# デシリアライズ検証 (後続の footer_crc も正しく復元される)
recovered_frame = TelemetryFrame.from_bytes(frame_bytes)
assert recovered_frame.payload == b"GPS_FIX_OK"
assert recovered_frame.readings == [100, 200, 300, 400]
assert recovered_frame.footer_crc == 0xCAFE
print("連動デシリアライズ成功！")

自動計算された payload_len: 10 (期待値: 10 bytes)
自動計算された item_count: 4 (期待値: 4 items)
シリアライズ結果 (23 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  0a 00 47 50 53 5f 46 49  58 5f 4f 4b 04 64 00 c8  |..GPS_FIX_OK.d..|
00000010  00 2c 01 90 01 fe ca                              |.,.....         |
  [Total: 23 bytes (`0x0017`)]
連動デシリアライズ成功！

## 3. `BinaryWriter.pad_to` による境界パディング

In [4]:
w = BinaryWriter()
w.write_uint32(0xDEADBEEF, name="magic")
w.write_cstring("boot", name="cmd")
print(f"pad_to 前のオフセット: {w.tell()} バイト")

w.pad_to(16, pad_byte=b"\xFF")
print(f"pad_to(16) 後のオフセット: {w.tell()} バイト")
print(f"パディング後のバッファ: {w.to_bytes().hex(' ')}")
assert len(w.to_bytes()) == 16

pad_to 前のオフセット: 9 バイト
pad_to(16) 後のオフセット: 16 バイト
パディング後のバッファ: ef be ad de 62 6f 6f 74 00 ff ff ff ff ff ff ff

## 4. インタラクティブ HTML 仕様書生成 (`write_html`)

ブラウザで開いてクリックできる、インタラクティブ Hex Inspector 付きの完全スタンドアロン HTML 仕様書を生成します。

In [5]:
sample_dir = Path("/home/ishii/PycharmProjects/binary_master/sample")
html_path = sample_dir / "telemetry_frame_spec.html"
frame.write_html(html_path, title="Telemetry Frame Protocol Spec")
print(f"スタンドアロン HTML 仕様書を生成しました: {html_path.name}")
print(f"HTML ファイルサイズ: {html_path.stat().st_size:,} bytes")
print("全 v0.3.0 機能の検証成功！")

スタンドアロン HTML 仕様書を生成しました: telemetry_frame_spec.html
HTML ファイルサイズ: 22,306 bytes
全 v0.3.0 機能の検証成功！